In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack

df = pd.read_csv('/content/anime.csv')

df.head()

df.info()
df.isnull().sum()

df.columns = df.columns.str.lower()

df['genre'] = df['genre'].fillna('')
df['rating'] = df['rating'].fillna(df['rating'].mean())

df['episodes'] = df['episodes'].replace('Unknown', np.nan)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

df = df.dropna(subset=['name'])

df = df.reset_index(drop=True)

print("\nAfter Cleaning:")
print(df.isnull().sum())

vectorizer = CountVectorizer(tokenizer=lambda x: x.split(', '))
genre_matrix = vectorizer.fit_transform(df['genre'])

scaler = MinMaxScaler()
numerical_features = scaler.fit_transform(df[['rating', 'episodes', 'members']])

feature_matrix = hstack([genre_matrix, numerical_features])

cosine_sim = cosine_similarity(feature_matrix, feature_matrix)

print("\nCosine similarity matrix shape:", cosine_sim.shape)

def recommend_anime(anime_name, top_n=5, threshold=0.4):

    if anime_name not in df['name'].values:
        return "Anime not found in dataset"

    index = df[df['name'] == anime_name].index[0]

    similarity_scores = list(enumerate(cosine_sim[index]))

    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similarity_scores = similarity_scores[1:]

    similarity_scores = [x for x in similarity_scores if x[1] >= threshold]

    similarity_scores = similarity_scores[:top_n]

    anime_indices = [i[0] for i in similarity_scores]

    return df[['name', 'genre', 'rating']].iloc[anime_indices]

print("\nRecommendations for Naruto:")
print(recommend_anime("Naruto", top_n=5, threshold=0.4))

print("\nThreshold = 0.3")
print(recommend_anime("Naruto", top_n=5, threshold=0.3))

print("\nThreshold = 0.6")
print(recommend_anime("Naruto", top_n=5, threshold=0.6))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB

After Cleaning:
anime_id     0
name         0
genre        0
type        25
episodes     0
rating       0
members      0
dtype: int64


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(



Cosine similarity matrix shape: (12294, 12294)

Recommendations for Naruto:
                                                   name  \
615                                  Naruto: Shippuuden   
1472        Naruto: Shippuuden Movie 4 - The Lost Tower   
1573  Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...   
486                            Boruto: Naruto the Movie   
1343                                        Naruto x UT   

                                                  genre  rating  
615   Action, Comedy, Martial Arts, Shounen, Super P...    7.94  
1472  Action, Comedy, Martial Arts, Shounen, Super P...    7.53  
1573  Action, Comedy, Martial Arts, Shounen, Super P...    7.50  
486   Action, Comedy, Martial Arts, Shounen, Super P...    8.03  
1343  Action, Comedy, Martial Arts, Shounen, Super P...    7.58  

Threshold = 0.3
                                                   name  \
615                                  Naruto: Shippuuden   
1472        Naruto: Shippuuden Movie 

In [3]:
'''
1. Difference between User-based and Item-based Collaborative Filtering

User-Based:
Finds similar users.
Recommends what similar users liked.

Item-Based:
Finds similar items.
Recommends similar items to what user liked.
Item-based is faster and more scalable.

2. What is Collaborative Filtering?

Collaborative filtering is a recommendation technique that uses user behavior (ratings, likes) to suggest items.

How it works:
Collect user-item interactions
Find similarity (users/items)
Recommend based on similar patterns
'''

'\n1. Difference between User-based and Item-based Collaborative Filtering\n\nUser-Based:\nFinds similar users.\nRecommends what similar users liked.\n\nItem-Based:\nFinds similar items.\nRecommends similar items to what user liked.\nItem-based is faster and more scalable.\n\n2. What is Collaborative Filtering?\n\nCollaborative filtering is a recommendation technique that uses user behavior (ratings, likes) to suggest items.\n\nHow it works:\nCollect user-item interactions\nFind similarity (users/items)\nRecommend based on similar patterns\n'